# route-map

**What does an answer look like?**

`shortest-vs-fewest` established that Dijkstra and BFS answer different
questions, and printed the difference. This experiment draws it.

Three pairs from the world network, each making a different point:

| Pair | Why |
|---|---|
| `HNL-BDL` | BFS saves a leg and spends 545 km. The classic trade. |
| `ANC-PVD` | The widest gap found: BFS pays 1,620 km for one fewer stop. |
| `SYD-JFK` | *Same* leg count, 7,057 km apart -- so hops cannot explain it. The two answers leave in opposite directions. |

Everything here uses `flight_planner.viz`, whose design and measurements are
in [`docs/visualization.md`](../../docs/visualization.md).

The data is pinned by `experiment.toml` and verified on load: if a byte of the
snapshot changes, this notebook raises instead of quietly producing a
different answer.

## Setup

Only this first cell differs between Colab and a local checkout.

`folium` and `pyproj` are **not** `flight_planner` dependencies -- the wheel
needs pandas and nothing else. They are notebook tools, so a notebook that
draws maps installs them itself. Locally `uv sync --group notebooks` (or
`make notebook`) has already done it.

In [ ]:
# In Colab, clone the repository and install the package plus the two
# mapping libraries:
#   !git clone https://github.com/<owner>/traiectoria-optima.git
#   %pip install -q ./traiectoria-optima
#   %pip install -q folium pyproj
# Then open this notebook from the clone.
try:
    import flight_planner  # noqa: F401
except ModuleNotFoundError as error:
    raise SystemExit("install the package first -- see the comment above") from error

In [ ]:
from pathlib import Path

from flight_planner import BFS, Dijkstra
from flight_planner.experiments import Experiment
from flight_planner.viz import MapExporter, NetworkLayer, RouteMap

# The notebook lives in the experiment directory, so the experiment is
# right here. Nothing resolves against a repository root.
experiment = Experiment.open(Path.cwd())
parameters = experiment.parameters

PAIRS = [tuple(pair) for pair in parameters['pairs']]
CONTEXT_PAIR = tuple(parameters['context_pair'])
FIGURES = parameters['figures']
PAIRS, CONTEXT_PAIR

## The data

Opening the snapshot re-hashes every file against the manifest. This is the
whole world: no narrowing, because the point of the third pair is a route that
crosses the Pacific.

In [ ]:
snapshot = experiment.snapshot
print(snapshot.snapshot_id, 'criteria:', snapshot.criteria or '(none -- the whole world)')

catalog = experiment.catalog()
planner = catalog.planner()
print(f'{len(catalog.airports):,} airports, {len(catalog.routes):,} routes')

## 1. The one-liner

`RouteMap.compare` is the 80% case: one pair, several algorithms, one coloured
layer each. The layer control carries the distance, so the legend states the
finding rather than naming a colour.

Colours come from the same palette `experiments/search-cost/plots.py` uses, so
Dijkstra is the same orange here as in that experiment's charts.

In [ ]:
origin, destination = PAIRS[0]

RouteMap.compare(
    planner,
    origin,
    destination,
    {'Dijkstra': Dijkstra(), 'BFS': BFS()},
)

Two lines, two answers. The southern line is BFS's two-leg
route; the northern one is Dijkstra's three legs, which is longer on the map
and shorter in kilometres -- the curve is why.

Note the lines are *curved*. A straight line between two coordinates is
straight in Web Mercator, which is neither the path flown nor the distance
`Route.distance_km` reports. `Geodesic` interpolates the great circle so the
picture agrees with the number beside it.

## 2. Building a map up by hand

`compare` is a named constructor over the fluent builder. The builder is there
when a question needs something `compare` does not offer -- here, all three
pairs on one map, each under Dijkstra only.

In [ ]:
route_map = RouteMap()

for pair in PAIRS:
    legs = planner.search_route(*pair, Dijkstra()).path if hasattr(
        planner, 'search_route'
    ) else planner.find_shortest_route(*pair, Dijkstra())[1]
    route_map.path(legs, name='-'.join(pair))

endpoints = [catalog.airport(code) for pair in PAIRS for code in pair]
route_map.airports(endpoints)

route_map

Each `.path()` call added one named, toggleable layer, and
`finish()` -- called for us by the notebook's renderer -- framed the map on
every point that was actually drawn.

That framing is the subtle part. Watch what happens with the Sydney route.

## 3. The antimeridian

`SYD-JFK` crosses the dateline. `Geod.npts` returns longitudes normalised into
`[-180, 180]`, so the raw interpolation jumps from `+179` to `-179` and Leaflet
draws a line straight back across the whole map.

`Geodesic` unwraps instead: it carries longitudes *past* 180 so the sequence
stays continuous. The consequence is that the drawn line lies outside the
bounding box of its own endpoints -- which is why `MapLayer` reports the points
it drew, and `RouteMap` frames itself on those rather than on airport
positions.

In [ ]:
dateline = RouteMap().path(
    planner.find_shortest_route('SYD', 'JFK', Dijkstra())[1], name='Dijkstra'
).airports([catalog.airport('SYD'), catalog.airport('JFK')])

dateline.finish()

longitudes = [lon for _, lon in dateline.drawn_points()]
(south, west), (north, east) = dateline.bounds()
print(f'drawn longitudes: {min(longitudes):8.1f} .. {max(longitudes):8.1f}')
print(f'fitted frame:     {west:8.1f} .. {east:8.1f}')
print(f'SYD is at {catalog.airport("SYD").longitude:.1f}, '
      f'JFK at {catalog.airport("JFK").longitude:.1f} -- '
      'a box built from those two would exclude the line entirely.')
dateline

One continuous line across the Pacific, and every marker on
it. The airport markers were moved into the same copy of the world as the
lines; left at their raw coordinates, JFK would render one globe to the west
and fall off the left edge of the frame.

Two details worth pausing on.

**The frame is 135 degrees wide, not 404.** `Geodesic` unwraps within a leg,
starting from that leg's own origin -- so `SYD-LAX` ends at 241.6 while
`LAX-JFK` begins again at its raw -118.4, a 360-degree jump between two legs
that are each individually correct. `PathLayer` shifts each leg into the copy
of the world the previous one finished in. Without that the map frames itself
over more than a full globe and zooms out past the route.

**Only Dijkstra is drawn here, deliberately.** BFS answers `SYD-JFK` in the
same two legs via Abu Dhabi -- `SYD-AUH-JFK`, 23,093 km against Dijkstra's
16,035 -- which means the two answers leave Sydney in *opposite directions*.
Drawn together they span the entire planet and neither is legible. That the
hop counts are identical is exactly why this pair needs a picture: no number
in the comparison table explains it.

The committed world snapshot has 647 route rows -- 137 distinct airport pairs
-- whose shorter great circle crosses the antimeridian, so none of this is an
edge case that can be deferred.

## 4. The network underneath, and what it costs

`.routes()` draws the whole network as context. It ships **switched off**, for
two reasons the next two cells show: it buries the answer, and it is most of
the file.

`NetworkLayer` collapses route rows to distinct airport pairs first. The route
table is keyed by flight number, so a busy city pair arrives three or four
times and would otherwise be painted over itself -- paying for it in file size
each time, and drawing it darker than the geometry warrants.

In [ ]:
layer = NetworkLayer(catalog.routes)
pairs_drawn = len(layer.distinct_pairs())

print(f'{len(catalog.routes):,} route rows')
print(f'{pairs_drawn:,} distinct airport pairs  '
      f'({len(catalog.routes) / pairs_drawn:.1f} rows per pair)')

In [ ]:
import tempfile

with_context = RouteMap.compare(
    planner,
    *CONTEXT_PAIR,
    {'Dijkstra': Dijkstra(), 'BFS': BFS()},
    context=catalog,
)
answer_only = RouteMap.compare(
    planner, *CONTEXT_PAIR, {'Dijkstra': Dijkstra(), 'BFS': BFS()}
)

# Measured into a temporary directory rather than into the repository: these
# two files are a measurement, not an artifact, and one is nearly 5 MB.
exporter = MapExporter()
sizes = {}
with tempfile.TemporaryDirectory() as scratch:
    for label, drawn in (('with context', with_context), ('answer only', answer_only)):
        written = exporter.html(drawn, Path(scratch) / f'{label}.html')
        sizes[label] = written.stat().st_size
        print(f'{label:14} {sizes[label] / 1024 / 1024:6.2f} MB')

print(f'\nthe context layer is {sizes["with context"] / sizes["answer only"]:,.0f}x '
      'the size of the answer it sits under')

Switch "All routes" on in the layer control below to see the
second reason. The answer is still there, but only just -- which is why the
honest default is off, and the reader turns it on when the question is "what
else was available?".

In [ ]:
with_context

## 5. A figure the report and the deck can cite

An inline map is gone when the kernel dies, and
`tests/experiments/test_committed.py` asserts this notebook carries no stored
output -- so nothing inline is ever committed. `MapExporter` writes the file
that is.

Both destinations get the same light PNG: there is no dark basemap to render
for the deck, because every Carto style is watermarked without an API key and
every Stadia style returns 401. The deck places this in a white card.

In [ ]:
FIGURES_TO_WRITE = {
    # The trade-off, and the picture docs/visualization.md opens with.
    'route-map-hnl-bdl.png': RouteMap.compare(
        planner, *CONTEXT_PAIR, {'Dijkstra': Dijkstra(), 'BFS': BFS()}
    ),
    # The geometry: one continuous line across the antimeridian, three
    # markers, and a frame 135 degrees wide rather than 404.
    'route-map-syd-jfk.png': RouteMap.compare(
        planner, 'SYD', 'JFK', {'Dijkstra': Dijkstra()}
    ),
}

for filename, drawn in FIGURES_TO_WRITE.items():
    for key in ('report_dir', 'slides_dir'):
        written = exporter.png(drawn, Path(FIGURES[key]) / filename)
    print(f'{filename}  {written.stat().st_size / 1024:,.0f} KB  '
          f'-> {FIGURES["report_dir"]} and {FIGURES["slides_dir"]}')

## The answer

Recorded next to the data that produced it. The numbers the map is *about* go
in `results.json`; the map itself is a file beside it, which keeps `record()`
free of opinions about artifacts it does not write.

In [ ]:
def measure(pair):
    """Return what each algorithm found for one pair."""
    found = {}
    for label, algorithm in (('Dijkstra', Dijkstra()), ('BFS', BFS())):
        _cost, legs = planner.find_shortest_route(*pair, algorithm)
        found[label] = {
            'km': round(sum(leg.distance_km for leg in legs), 1),
            'legs': len(legs),
            'route': '-'.join([legs[0].origin.iata_code]
                              + [leg.destination.iata_code for leg in legs]),
        }
    found['km_penalty'] = round(found['BFS']['km'] - found['Dijkstra']['km'], 1)
    found['legs_saved'] = found['Dijkstra']['legs'] - found['BFS']['legs']
    return found


comparison = {'-'.join(pair): measure(pair) for pair in PAIRS}
for pair, found in comparison.items():
    print(f"{pair}: Dijkstra {found['Dijkstra']['km']:>9,.0f} km / "
          f"{found['Dijkstra']['legs']} legs | "
          f"BFS {found['BFS']['km']:>9,.0f} km / {found['BFS']['legs']} legs | "
          f"BFS pays {found['km_penalty']:>8,.0f} km")

In [ ]:
experiment.record(
    {
        'comparison': comparison,
        'network': {
            'route_rows': len(catalog.routes),
            'distinct_pairs': pairs_drawn,
        },
        'figures': sorted(FIGURES_TO_WRITE),
    },
    catalog=catalog,
)